<img src="logo.png" alt="Vegeta" width="240">

# Quadcopter propeller — performance, propulsion system and what it hands on

The 5-inch tri-blade propeller of the quadcopter from notebook 08, on its 2306 motor and 4S battery.
Blade element momentum theory (`vegeta.boreas`) gives thrust, torque and power over rpm and airspeed;
the motor model turns throttle into rpm and current; the battery gives hover time. Everything is drawn,
and the important numbers are **exported to one JSON file** (`_runs/propeller/quad_5x43.json`) plus a
propeller STL, for the mission, vibration and fatigue work that comes next.

Assumptions are explicit and coarse: a generic blade section (no measured polar, no Reynolds effect),
axial inflow only, rigid blades, a first-order motor model. Treat the results as ±25 % until you fit the
section to your propeller's data.

In [ ]:
import json, math, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from vegeta import boreas, dedalus
from vegeta.dedalus import viz as dviz
from vegeta.dedalus.examples import Propeller as PropellerCAD

RUNS = Path("_runs/propeller"); RUNS.mkdir(parents=True, exist_ok=True)
RHO = 1.2

# ---- hand-copied from notebook 08 (mass budget): keep these in sync by hand, on purpose -----------
AUW_KG = 0.476                 # all-up weight with the preferred frame
MOTORS = 4
HOVER_THRUST_PER_MOTOR = AUW_KG * 9.81 / MOTORS

## 1. The hardware (your inputs)

In [ ]:
d, p = boreas.inches(5, 4.3)
prop = boreas.Propeller.from_pitch("5x4.3 tri-blade", d, p, blades=3, chord_root_m=0.010, chord_max_m=0.016,
                                   chord_tip_m=0.006, mass_kg=0.0045, rotor_mass_kg=0.020,   # rotor = prop + motor bell
                                   notes="generic planform; fit chord/beta to the real propeller for better numbers")
airfoil = boreas.Airfoil(name="thin cambered section", cl_alpha=2 * math.pi * 0.9, alpha0_deg=-3.0, cl_max=1.1, cd0=0.025, k=0.045,
                         source="assumed for a moulded 5-inch blade at Re ~ 1e5")
motor = boreas.Motor("2306-2400KV", kv_rpm_per_volt=2400, resistance_ohm=0.06, no_load_current_a=1.2, max_current_a=40, mass_kg=0.030)
battery = boreas.Battery("4S 1500 mAh", cells=4, capacity_ah=1.5, usable_fraction=0.8, mass_kg=0.180)
system = boreas.Propulsion(prop, airfoil, motor, battery, rho=RHO)
pd.DataFrame(prop.describe()["stations"]).set_index("r_m").T

## 2. The propeller, drawn

Planform and blade angle from the station table, and the same planform built as a solid in Dedalus
(`vegeta.dedalus.examples.Propeller` uses the same formula): sections at three radii, a 3D view, and an
STL export for a later CFD or print.

In [ ]:
r = np.array(prop.r); c = np.array(prop.chord); beta = np.array(prop.beta_deg)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].fill_between(r * 1000, -c * 1000 * 0.3, c * 1000 * 0.7, color="#9fb8d0"); ax[0].set_aspect("equal")
ax[0].set(xlabel="radius [mm]", ylabel="chord [mm]", title=f"planform, solidity {prop.solidity:.3f}"); ax[0].grid(alpha=0.3)
ax[1].plot(r * 1000, beta, "o-"); ax[1].set(xlabel="radius [mm]", ylabel="blade angle β [deg]", title=f"twist for {p * 1000:.0f} mm pitch"); ax[1].grid(alpha=0.3)
fig.tight_layout()

In [ ]:
cad = PropellerCAD().generate(diameter=d * 1000, pitch=p * 1000, blades=3, hub_diameter=12, hub_height=7, bore=5,
                              chord_root=10, chord_max=16, chord_tip=6, thickness=0.10, camber=0.05)
prop_files = cad.export(RUNS / "quad_5x43_cad", formats=("step", "stl"), stl_tolerance=0.02)
print(f"CAD volume {cad.volume:.0f} mm^3 -> {cad.volume * 1.2e-3:.1f} g at 1.2 g/cm^3 (input mass_kg = {prop.mass_kg * 1000:.1f} g)")
dviz.show(dviz.plot3d(cad))

In [ ]:
fig = dviz.plot_sections(cad, normal="x", positions=[20.0, 40.0, 58.0], cols=3)   # blade sections along +X

## 3. Static performance (hover): thrust and power against rpm

In [ ]:
rpms = np.linspace(4000, 30000, 27)
static = [boreas.solve(prop, airfoil, rpm, 0.0, RHO) for rpm in rpms]
T = np.array([o.thrust for o in static]); P = np.array([o.power for o in static])
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
ax[0].plot(rpms, T); ax[0].axhline(HOVER_THRUST_PER_MOTOR, ls="--", color="#c62828", label="hover thrust / motor"); ax[0].legend()
ax[0].set(xlabel="rpm", ylabel="thrust [N]", title="static thrust")
ax[1].plot(rpms, P); ax[1].set(xlabel="rpm", ylabel="shaft power [W]", title="static power")
ax[2].plot(rpms, [o.figure_of_merit for o in static]); ax[2].set(xlabel="rpm", ylabel="figure of merit", ylim=(0, 1), title="hover efficiency")
for a in ax: a.grid(alpha=0.3)
fig.tight_layout()
print(f"at 20 000 rpm: Ct {static[np.argmin(abs(rpms - 20000))].ct:.4f}, Cp {static[np.argmin(abs(rpms - 20000))].cp:.4f}, "
      f"tip Mach {static[-1].tip_mach:.2f} at {rpms[-1]:.0f} rpm")

In [ ]:
hover_aero = boreas.rpm_for_thrust(prop, airfoil, HOVER_THRUST_PER_MOTOR, 0.0, RHO)
fig, ax = plt.subplots(1, 3, figsize=(14, 3.4))
ax[0].plot(hover_aero.r * 1000, hover_aero.dT_dr); ax[0].set(xlabel="radius [mm]", ylabel="dT/dr [N/m]", title=f"thrust loading at hover ({hover_aero.rpm:.0f} rpm)")
ax[1].plot(hover_aero.r * 1000, hover_aero.alpha_deg); ax[1].set(xlabel="radius [mm]", ylabel="angle of attack [deg]", title="section incidence")
ax[2].plot(hover_aero.r * 1000, hover_aero.induced_velocity); ax[2].set(xlabel="radius [mm]", ylabel="induced velocity [m/s]", title="inflow")
for a in ax: a.grid(alpha=0.3)
fig.tight_layout()

## 4. Motor + battery: throttle → rpm, current, power

The motor curve meets the propeller torque curve at each throttle; current above the motor limit is
flagged, never hidden.

In [ ]:
throttles = np.linspace(0.2, 1.0, 17)
sweep = system.sweep(throttles, airspeed=0.0)
sw = pd.DataFrame([{"throttle": s.throttle, "rpm": s.rpm, "thrust_N": s.thrust, "current_A": s.current,
                    "electrical_W": s.electrical_power, "motor_eff": s.motor_efficiency, "current_limited": s.current_limited} for s in sweep])
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
ax[0].plot(sw.throttle, sw.thrust_N, "o-"); ax[0].axhline(HOVER_THRUST_PER_MOTOR, ls="--", color="#c62828"); ax[0].set(xlabel="throttle", ylabel="thrust [N]")
ax[1].plot(sw.throttle, sw.current_A, "o-"); ax[1].axhline(motor.max_current_a, ls="--", color="#c62828", label="motor limit"); ax[1].legend(); ax[1].set(xlabel="throttle", ylabel="current [A]")
ax[2].plot(sw.thrust_N, sw.electrical_W / sw.thrust_N, "o-"); ax[2].set(xlabel="thrust [N]", ylabel="W per N", title="electrical power per newton")
for a in ax: a.grid(alpha=0.3)
fig.tight_layout()
sw.round(3)

In [ ]:
hover = system.for_thrust(HOVER_THRUST_PER_MOTOR)
punch = system.at_throttle(1.0)
cruise = system.for_thrust(HOVER_THRUST_PER_MOTOR * 1.3)        # forward flight at ~40 deg tilt: 1/cos(40) ~ 1.3
hover_minutes = battery.usable_wh / (MOTORS * hover.electrical_power) * 60
summary = pd.DataFrame({
    "hover": {"throttle": hover.throttle, "rpm": hover.rpm, "thrust_N": hover.thrust, "current_A": hover.current, "electrical_W": hover.electrical_power},
    "cruise (tilted)": {"throttle": cruise.throttle, "rpm": cruise.rpm, "thrust_N": cruise.thrust, "current_A": cruise.current, "electrical_W": cruise.electrical_power},
    "full throttle": {"throttle": 1.0, "rpm": punch.rpm, "thrust_N": punch.thrust, "current_A": punch.current, "electrical_W": punch.electrical_power},
}).round(2)
print(f"thrust-to-weight at full throttle: {MOTORS * punch.thrust / (AUW_KG * 9.81):.2f}"
      + ("  (CURRENT LIMITED: the motor cannot deliver this; the ESC/motor limit decides)" if punch.current_limited else ""))
print(f"hover endurance: {hover_minutes:.1f} min on {battery.usable_wh:.0f} Wh usable")
summary

## 5. What the frame will feel: excitation frequencies and forces

Per motor: shaft frequency (1P), blade-pass frequency (3P for three blades) and the rotating unbalance
force for an ISO balance grade G 6.3 (a typical hobby-grade prop; G 2.5 after balancing). These are the
inputs for the vibration and fatigue notebooks: a frame mode near one of these lines is a problem.

In [ ]:
rr = np.linspace(5000, 30000, 26)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(rr, rr / 60, label="1P (shaft)"); ax[0].plot(rr, prop.blades * rr / 60, label=f"{prop.blades}P (blade pass)")
for name, pt in (("hover", hover), ("full", punch)):
    ax[0].axvline(pt.rpm, color="#888", ls=":"); ax[0].text(pt.rpm, 50, name, rotation=90, va="bottom")
ax[0].set(xlabel="rpm", ylabel="frequency [Hz]", title="excitation lines"); ax[0].legend(); ax[0].grid(alpha=0.3)
for g in (6.3, 2.5):
    ax[1].plot(rr, [boreas.unbalance_force(prop.rotor_mass_kg, x, g) for x in rr], label=f"G {g}")
ax[1].set(xlabel="rpm", ylabel="rotating force [N]", title=f"unbalance force, rotor {prop.rotor_mass_kg * 1000:.0f} g"); ax[1].legend(); ax[1].grid(alpha=0.3)
fig.tight_layout()
pd.DataFrame({k: boreas.excitations(prop, v.rpm) for k, v in (("hover", hover), ("cruise", cruise), ("full", punch))}).round(2)

## 6. Axial inflow: climbing and descending

Thrust falls with axial speed (climb); the map covers 0–15 m/s. A vertical descent runs into the
vortex-ring regime that momentum theory cannot describe, so the map stops at zero airspeed.

In [ ]:
grid = boreas.performance_map(prop, airfoil, np.linspace(5000, 30000, 11), np.linspace(0, 15, 6), RHO)
Tg = np.array(grid["thrust_n"])
fig, ax = plt.subplots(figsize=(6, 3.8))
for j, v in enumerate(grid["airspeed_m_s"]):
    ax.plot(grid["rpm"], Tg[:, j], label=f"{v:.0f} m/s")
ax.set(xlabel="rpm", ylabel="thrust [N]", title="thrust vs rpm at axial airspeeds"); ax.legend(ncol=2); ax.grid(alpha=0.3)
print("unconverged map points:", grid["unconverged_points"])

## 7. Export — the hand-off

One JSON with: propeller geometry, the section model, motor and battery, the rpm × airspeed map, and the
named operating points **hover / cruise / full** each with its excitation summary (rpm, 1P, blade-pass,
unbalance force). Later notebooks copy from this file by hand (mission segments, vibration loads); the
STL/STEP are for CFD or printing.

In [ ]:
res = boreas.export(RUNS / "quad_5x43.json", prop, airfoil, map=grid, motor=motor, battery=battery,
                    points={"hover": hover, "cruise": cruise, "full": punch},
                    notes=f"quadcopter from notebook 08; AUW {AUW_KG} kg, {MOTORS} motors; hover endurance {hover_minutes:.1f} min")
print(res)
doc = boreas.load(res.artifacts["json"])
pd.DataFrame({k: {"rpm": v["rpm"], "thrust_N": v["aero"]["thrust"], "current_A": v["current"],
                  "shaft_hz": v["excitation"]["shaft_hz"], "blade_pass_hz": v["excitation"]["blade_pass_hz"],
                  "unbalance_N": v["excitation"]["unbalance_force_n"]} for k, v in doc["points"].items()}).round(2)

In [ ]:
print("files for the next steps:")
for f in sorted(RUNS.glob("quad_5x43*")):
    print("  ", f, f"({f.stat().st_size / 1024:.0f} kB)" if f.is_file() else "")
print("\nkeys in the JSON:", list(doc))
print("points:", list(doc["points"]), "| map:", len(doc["map"]["rpm"]), "rpm x", len(doc["map"]["airspeed_m_s"]), "airspeeds")

## 8. CFD check of the hover point — OpenFOAM, rotating reference frame

Blade element theory says 1.17 N at the hover rpm. `aeromant`'s `rotor_mrf_static` template puts the
same CAD propeller in a rotating cell zone (MRF, steady k-ω SST) and integrates the blade forces.
It is a *check*, coarse by design (about 100 k cells, a few minutes on one core; `surface_level=4`
for a finer blade at several times the cost); expect tens of percent against BEMT, and read the sign
message: a negative thrust means the propeller is handed against `rotation`.

The propeller CAD has its axis along Z; the template wants it along +x, so the shape is rotated first.
Nothing runs unless you run the cell; `VEGETA_SKIP_OPENFOAM=1` skips it (used when the notebooks are
executed headlessly on a machine without OpenFOAM).

In [ ]:
import os
from vegeta import aeromant
from vegeta.aeromant import viz as aviz

RUN_CFD = os.environ.get("VEGETA_SKIP_OPENFOAM") != "1"
prop_x = dedalus.Geometry.from_cadquery(cad.shape.rotate((0, 0, 0), (0, 1, 0), 90), name="prop_axis_x")   # axis z -> +x
stl_x = prop_x.export_stl(RUNS / "quad_5x43_cad" / "prop_axis_x.stl", tolerance=0.02)
CFD_PARAMS = dict(rpm=hover.rpm, diameter=d, kinematic_viscosity=1.5e-5, density=RHO, rotation=1, iterations=400,
                  cells_per_diameter=6.0, surface_level=3, near_level=2, rotor_level=2, wake_level=1)   # coarse: minutes, not hours
cfd = None
if RUN_CFD:
    case = aeromant.CFDCase("rotor_mrf_static", stl_x, CFD_PARAMS, workdir=RUNS / "quad_5x43_cfd_hover", geometry_units="mm",
                            environment=aeromant.OpenFOAMEnvironment.detect())
    print(case.prepare(overwrite=True))
    cfd = case.run(progress=True)
    print(cfd)
else:
    print("CFD skipped (VEGETA_SKIP_OPENFOAM=1): run this cell on a machine with OpenFOAM to get the check")

In [ ]:
if cfd is not None and cfd.ok:
    m = cfd.metrics
    compare = pd.DataFrame({"BEMT (Boreas)": {"thrust_N": hover.thrust, "torque_Nm": hover.aero.torque, "power_W": hover.aero.power, "figure_of_merit": hover.aero.figure_of_merit},
                            "CFD (rotor_mrf_static)": {"thrust_N": m["thrust_N"], "torque_Nm": m["torque_Nm"], "power_W": m["power_W"], "figure_of_merit": m["figure_of_merit"]}})
    compare["CFD / BEMT"] = compare["CFD (rotor_mrf_static)"] / compare["BEMT (Boreas)"]
    print(f"{m['mesh_cells']} cells, {'converged' if m['converged'] else 'not converged'} in {m['iterations']} iterations; forces averaged over the last {m['averaging_window']}")
    display(compare.round(3))

In [ ]:
if cfd is not None and cfd.ok:
    aviz.show(aviz.plot_field_slice(case, "U", normal="z"))          # the plane through the axis: inflow above, slipstream below
    fig = aviz.plot_section(case, "p", normal="z", zoom=2)

In [ ]:
if cfd is not None and cfd.ok:
    aviz.show(aviz.plot_streamlines(case, n=80, normal_plane="z"))
    aviz.show(aviz.plot_surface_pressure(case))

In [ ]:
if cfd is not None and cfd.ok:                                    # add the CFD point to the exported JSON
    doc = json.loads((RUNS / "quad_5x43.json").read_text())
    doc["cfd_hover"] = {"template": "rotor_mrf_static", "parameters": CFD_PARAMS, "metrics": {k: v for k, v in cfd.metrics.items() if not isinstance(v, (list, dict))}}
    (RUNS / "quad_5x43.json").write_text(json.dumps(doc, indent=2, default=float))
    print("cfd_hover added to", RUNS / "quad_5x43.json")

**Next:** the mission notebooks read `hover`/`cruise`/`full` (rpm, thrust, current) to build flight
segments, and the excitation lines to check the frame's natural frequencies and to build the vibratory
load spectrum for fatigue. Fit `airfoil` to a measured polar of your propeller before trusting the
absolute numbers.